# California Housing Market : Feature engineering and feature selection
In the previous exercise, we concluded it was worth including more variables in a model. But is this set of variables **the best** we could have chosen ? In this exercises, we'll go further by applying two canonical methods:
* Feature engineering consists in creating more variables from the original dataset
* Feature selection allows to select the best set of features among all the available variables

## The dataset
1. Load the California Housing dataset again and remove the outliers:

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

In [2]:
from sklearn import datasets
data = datasets.fetch_california_housing(data_home=None, download_if_missing=True, return_X_y=False)
print(data.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [4]:
data_pd = pd.DataFrame(data=data.data, columns=data.feature_names)
data_pd["Price"] = pd.DataFrame(data=data.target)
data_pd.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
mask_outliers = (data_pd["AveRooms"] < 10) & (data_pd["AveBedrms"] < 10) & (data_pd["Population"] < 15000) & (data_pd["AveOccup"] < 10) & (data_pd["Price"] < 5)
data_filtered = data_pd[mask_outliers]
data_filtered.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [6]:
print("Number orf rows:", len(data_filtered))
print()
print("Display of dataset:")
print(data_filtered.head())
print()
print("Basics statistics:")
print(data_filtered.describe())
print()
print("Percentage of missing values:")
perc = 100 * data_filtered.isnull().sum()/ len(data_filtered)
print(perc)

Number orf rows: 19398

Display of dataset:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  Price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  

Basics statistics:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  19398.000000  19398.000000  19398.000000  19398.000000  19398.000000   
mean       3.674497     28.496907      5.210648      1.066038   1442.172080   
std        1.563397     12.477953      1.168098      0.128846   1077.498768   
min        0.499900      1.0000

2. Separate the target from the features

In [9]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude']
target_variable = "Price"

X = data_filtered.loc[:,features_list]
Y = data_filtered.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: Price, dtype: float64

X :
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  
3    -122.25  
4    -122.25  


## From linear to non-linear regression
An easy way of implementing a non-linear regression is to create by hand more columns containing non-linear functions of the features.

3. For each explanatory variable, create 3 new columns in $X$ containing the following functions:
* $\textrm{X}^2$
* $\textrm{X}^3$
* $\textrm{X}^4$
* $\frac{1}{\textrm{X}}$
* $\frac{1}{\textrm{X}^2}$

In [10]:
for col in features_list:
    data_filtered[f'{col}_2'] = data_filtered[col] ** 2
    data_filtered[f'{col}_3'] = data_filtered[col] ** 3
    data_filtered[f'{col}_4'] = data_filtered[col] ** 4
    data_filtered[f'{col}_inverse'] = 1 / data_filtered[col]
    data_filtered[f'{col}_inverse2'] = 1 / (data_filtered[col] ** 2)

C:\Users\briic\AppData\Local\Temp\ipykernel_24664\3810268415.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\briic\AppData\Local\Temp\ipykernel_24664\3810268415.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\briic\AppData\Local\Temp\ipykernel_24664\3810268415.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/panda

In [11]:
data_filtered.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price,MedInc_2,...,Latitude_2,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,69.308955,...,1434.8944,54353.799872,2.058922e+06,0.026399,0.000697,14940.1729,-1.826137e+06,2.232088e+08,-0.008181,0.000067
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,68.913242,...,1433.3796,54267.751656,2.054577e+06,0.026413,0.000698,14937.7284,-1.825689e+06,2.231357e+08,-0.008182,0.000067
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,52.669855,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14942.6176,-1.826586e+06,2.232818e+08,-0.008181,0.000067
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,31.844578,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,14.793254,...,1432.6225,54224.761625,2.052407e+06,0.026420,0.000698,14945.0625,-1.827034e+06,2.233549e+08,-0.008180,0.000067


4. Split your dataset into train (80%) and test (20%)

In [22]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = data_filtered.drop("Price", axis=1).columns.tolist()
target_variable = "Price"

X = data_filtered.loc[:,features_list]
Y = data_filtered.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: Price, dtype: float64

X :
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude   MedInc_2    MedInc_3  ...  Latitude_2    Latitude_3  \
0    -122.23  69.308955  577.010912  ...   1434.8944  54353.799872   
1    -122.22  68.913242  572.076387  ...   1433.3796  54267.751656   
2    -122.24  52.669855  382.246204  ...   1432.6225  54224.761625   
3    -122.25  31.844578  179.702136  ...   1432.6225  54224.761625   
4    -122.25  14.793254   56.897815  ...   1432.6225  5

In [23]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



5. Apply the same preprocessing as in the previous exercise

In [24]:
print("Preprocessing X_train...")
print(X_train.head())
print()
preprocessor = Pipeline(steps=[
    ('scaler', StandardScaler())
])
X_train = preprocessor.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array
print() 

Preprocessing X_train...
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
3235   2.3889       6.0  6.316614   1.294671       992.0  3.109718     36.09   
13981  3.4912       7.0  8.355308   1.554795      2933.0  2.511130     34.85   
9219   1.9464      36.0  4.975510   1.053061       639.0  2.608163     37.12   
10851  3.1667      22.0  3.803838   1.000000      1952.0  2.081023     33.66   
8888   4.2520      31.0  3.978296   1.039389      1985.0  1.595659     34.03   

       Longitude   MedInc_2   MedInc_3  ...  Latitude_2    Latitude_3  \
3235     -119.57   5.706843  13.633078  ...   1302.4881  47006.795529   
13981    -117.46  12.188477  42.552412  ...   1214.5225  42326.109125   
9219     -120.27   3.788473   7.373884  ...   1377.8944  51147.440128   
10851    -117.90  10.027989  31.755632  ...   1132.9956  38136.631896   
8888     -118.49  18.079504  76.874051  ...   1158.0409  39408.131827   

         Latitude_4  Latitude_inverse  Latitude_inverse

In [25]:
# Test pipeline
print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = preprocessor.transform(X_test)
print("...Done!")
print(X_test[0:5,:]) # X_test is now a numpy array
print() 

Preprocessing X_test...
       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
17333  5.2990      12.0  7.214932   1.047511      1200.0  2.714932     34.91   
1012   2.6667      44.0  4.541284   1.027523       277.0  2.541284     37.68   
5124   1.5521      30.0  3.850679   1.002262      1966.0  4.447964     33.99   
1845   6.3538      49.0  6.293886   1.017751      1148.0  2.264300     37.90   
4035   3.2154      20.0  4.133444   1.060181      7450.0  1.772122     34.17   

       Longitude   MedInc_2    MedInc_3  ...  Latitude_2    Latitude_3  \
17333    -120.44  28.079401  148.792746  ...   1218.7081  42545.099771   
1012     -121.77   7.111289   18.963674  ...   1419.7824  53497.400832   
5124     -118.26   2.409014    3.739031  ...   1155.3201  39269.330199   
1845     -122.28  40.370774  256.507827  ...   1436.4100  54439.939000   
4035     -118.52  10.338797   33.243368  ...   1167.5889  39896.512713   

         Latitude_4  Latitude_inverse  Latitude_in

6. Train a model including all these features. Do you get better performances than before?

In [26]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Train model...
...Done.


In [27]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

Predictions on training set...
...Done.
[0.81154535 1.38885205 0.90064246 ... 2.38237549 2.48941613 1.28639375]



In [28]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[3.04601756 1.62362395 1.18009908 ... 2.22226679 2.29221063 2.9870925 ]



In [29]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

R2 score on training set :  0.6831497565034788
R2 score on test set :  0.6858617135623264


## Forward selection
This feature engineering trick improved the model's score significantly ! But now, the model is a lot more complex as it uses 32 input features. Do we really need all these features? Let's implement the forward selection method described in this morning's lecture. 

Fortunately, the latest versions of sklearn provide a class that implements forward selection, such that we don't need to code the algorithm by hand 🥳

7. Have a look at the documentation of [SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html) and try to understand the following lines of code:

In [30]:
from sklearn.feature_selection import  SequentialFeatureSelector
feature_selector =  SequentialFeatureSelector(regressor, n_features_to_select = 20)
feature_selector.fit(X_train, Y_train)
features_list = X.columns
best_features = features_list[feature_selector.support_]
print("According to the forward selection algorithm, the following features should be kept: ")
print(best_features.to_list())

According to the forward selection algorithm, the following features should be kept: 
['MedInc', 'HouseAge', 'Population', 'Latitude', 'MedInc_inverse2', 'AveRooms_3', 'AveRooms_4', 'AveRooms_inverse', 'AveRooms_inverse2', 'AveBedrms_inverse', 'Population_2', 'Population_inverse2', 'AveOccup_3', 'AveOccup_inverse', 'AveOccup_inverse2', 'Latitude_3', 'Latitude_4', 'Latitude_inverse', 'Latitude_inverse2', 'Longitude_inverse']


8. Create a DataFrame X_best containing only the best set of features, train a model only with these features and evaluate the performances

In [31]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = best_features.to_list()
target_variable = "Price"

X = data_filtered.loc[:,features_list]
Y = data_filtered.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: Price, dtype: float64

X :
   MedInc  HouseAge  Population  Latitude  MedInc_inverse2  AveRooms_3  \
0  8.3252      41.0       322.0     37.88         0.014428  340.671954   
1  8.3014      21.0      2401.0     37.86         0.014511  242.753076   
2  7.2574      52.0       496.0     37.85         0.018986  569.338486   
3  5.6431      52.0       558.0     37.85         0.031403  196.868367   
4  3.8462      52.0       565.0     37.85         0.067598  247.892488   

    AveRooms_4  AveRooms_inverse  AveRooms_inverse2  AveBedrms_inverse  \
0  2379.296184          0.143182           0.020501           0.976744   
1  1514.326968          0.160304           0.025697           1.028933   
2  4718.754574          0.120654           0.014557           0.931579   
3  1145.252511          0.171900           0.029549           0.931915   
4  1557.224240          0.159189           0.02

In [32]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

print("Preprocessing X_train...")
print(X_train.head())
print()
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_train = preprocessor.fit_transform(X_train)
print("...Done!")
print(X_train[0:5,:]) # X_train is now a numpy array
print()

# Test pipeline
print("Preprocessing X_test...")
print(X_test.head())
print()
X_test = preprocessor.transform(X_test)
print("...Done!")
print(X_test[0:5,:]) # X_test is now a numpy array
print() 

# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train)
print("...Done.")
print(Y_train_pred)
print()

# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Dividing into train and test sets...
...Done.

Preprocessing X_train...
       MedInc  HouseAge  Population  Latitude  MedInc_inverse2  AveRooms_3  \
3235   2.3889       6.0       992.0     36.09         0.175228  252.030501   
13981  3.4912       7.0      2933.0     34.85         0.082045  583.293888   
9219   1.9464      36.0       639.0     37.12         0.263959  123.172247   
10851  3.1667      22.0      1952.0     33.66         0.099721   55.038428   
8888   4.2520      31.0      1985.0     34.03         0.055311   62.963842   

        AveRooms_4  AveRooms_inverse  AveRooms_inverse2  AveBedrms_inverse  \
3235   1591.979495          0.158313           0.025063           0.772397   
13981  4873.600216          0.119684           0.014324           0.643172   
9219    612.844771          0.200984           0.040395           0.949612   
10851   209.357262          0.262892           0.069112           1.000000   
8888    250.488789          0.251364           0.063184           0.9

## Advanced feature engineering
Let's make even more advanced feature engineering. Until now, we've included the latitude and longitude as such into the models. However, usually the GPS coordinates are not used rawly, instead we deduce some geographical information from these. Let's use an API that will allows to retrieve the name of the city from the latitude and longitude.

💡 As the calls to the API may be time-consuming, we'll work on a sample of the dataset.

9. Take a sample of your dataset X (the one that contains all the features and not only the best set, because we need the values of Latitude and Longitude). Keep only 150 rows.

In [36]:
# Separate target variable Y from features X
print("Separating labels from features...")
features_list = data_filtered.drop("Price", axis=1).columns.tolist()
target_variable = "Price"

X_sample = data_filtered.loc[:150,features_list]
Y_sample = data_filtered.loc[:150,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y_sample.head())
print()
print('X :')
print(X_sample.head())

Separating labels from features...
...Done.

Y : 
0    4.526
1    3.585
2    3.521
3    3.413
4    3.422
Name: Price, dtype: float64

X :
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude   MedInc_2    MedInc_3  ...  Latitude_2    Latitude_3  \
0    -122.23  69.308955  577.010912  ...   1434.8944  54353.799872   
1    -122.22  68.913242  572.076387  ...   1433.3796  54267.751656   
2    -122.24  52.669855  382.246204  ...   1432.6225  54224.761625   
3    -122.25  31.844578  179.702136  ...   1432.6225  54224.761625   
4    -122.25  14.793254   56.897815  ...   1432.6225  5

10. Create a Y_sample variable containing the target values corresponding to the rows that were kept in X_sample

1523     3.068
2704     0.675
2047     1.272
11174    2.052
1763     1.341
Name: Price, dtype: float64

11. Use the following help to translate the longitude and latitude of the data to find the cities corresponding to each observation: [geopy](https://pypi.org/project/geopy)

In [35]:
!pip install geopy


   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]



In [37]:
# Example of how to get the adress from a given pair of latitude/longitude coordinates
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="yet_another_app")
location = geolocator.reverse("52.509669, 13.376294")
loc_dict = dict(location.raw)
loc_dict["address"]

{'house_number': '11',
 'road': 'Potsdamer Platz',
 'suburb': 'Tiergarten',
 'borough': 'Mitte',
 'city': 'Berlin',
 'ISO3166-2-lvl4': 'DE-BE',
 'postcode': '10785',
 'country': 'Deutschland',
 'country_code': 'de'}

In [38]:
# Use geopy to extract the city of each row in the sample dataset
X_sample["City"] = 0
for i, row in X_sample.iterrows():
    geolocator = Nominatim(user_agent="yet_another_app_2")
    location = geolocator.reverse("{}, {}".format(X_sample.loc[i, "Latitude"], X_sample.loc[i, "Longitude"]), 
                                  timeout = None)
    loc_dict = dict(location.raw)
    try:
        X_sample.loc[i, "City"] = loc_dict["address"]["city"]
    except:
        try:
            X_sample.loc[i, "City"] = loc_dict["address"]["town"]
        except:
            try:
                X_sample.loc[i, "City"] = loc_dict["address"]["village"]
            except:
                pass
# If city was not found, replace by "Unknown"
X_sample.loc[X_sample['City'] == 0, 'City'] = "Unknown"

C:\Users\briic\AppData\Local\Temp\ipykernel_24664\1970226461.py:9: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Oakland' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.



In [39]:
X_sample.describe(include='all')

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedInc_2,MedInc_3,...,Latitude_3,Latitude_4,Latitude_inverse,Latitude_inverse2,Longitude_2,Longitude_3,Longitude_4,Longitude_inverse,Longitude_inverse2,City
count,149.000000,149.000000,149.000000,149.000000,149.000000,149.000000,149.000000,149.000000,149.000000,149.000000,...,149.000000,1.490000e+02,149.000000,1.490000e+02,149.000000,1.490000e+02,1.490000e+02,149.000000,1.490000e+02,149
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Oakland
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,137
mean,3.218595,43.000000,4.970076,1.081219,900.335570,2.379286,37.825503,-122.255705,15.106081,91.351659,...,54119.577559,2.047101e+06,0.026437,6.989254e-04,14946.458117,-1.827290e+06,2.233967e+08,-0.008180,6.690550e-05,NaN
std,2.186047,11.585172,1.382896,0.122885,573.607392,0.549293,0.016621,0.028123,20.474284,185.439747,...,71.352277,3.598806e+03,0.000012,6.141275e-07,6.875713,1.260782e+03,2.054991e+05,0.000002,3.078887e-08,NaN
min,0.499900,2.000000,1.714286,0.571429,18.000000,1.437141,37.790000,-122.300000,0.249900,0.124925,...,53967.298139,2.039424e+06,0.026399,6.969154e-04,14927.952400,-1.829277e+06,2.228438e+08,-0.008185,6.685703e-05,NaN
25%,1.687500,36.000000,3.980237,1.023810,534.000000,2.101083,37.810000,-122.280000,2.847656,4.805420,...,54053.028541,2.043745e+06,0.026427,6.983896e-04,14942.617600,-1.828379e+06,2.232818e+08,-0.008181,6.687890e-05,NaN
50%,2.600000,49.000000,4.797980,1.068000,756.000000,2.346154,37.820000,-122.260000,6.760000,17.576000,...,54095.927768,2.045908e+06,0.026441,6.991284e-04,14947.507600,-1.827482e+06,2.234280e+08,-0.008179,6.690079e-05,NaN
75%,3.964300,52.000000,6.047244,1.114943,1129.000000,2.606880,37.840000,-122.240000,15.715674,62.301648,...,54181.794304,2.050239e+06,0.026448,6.994983e-04,14952.398400,-1.826586e+06,2.235742e+08,-0.008178,6.692268e-05,NaN


12. Make a train/test splitting from X_sample and Y_sample

In [49]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X_sample, Y_sample, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



13. What preprocessings are necessary now ? The cells below implement the preprocessings, read it carefully and check what is done

In [50]:
categorical_features = ['City']
numeric_features = [c for c in X_sample.columns if c != 'City']

In [51]:
print(categorical_features)
print(numeric_features)

['City']
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedInc_2', 'MedInc_3', 'MedInc_4', 'MedInc_inverse', 'MedInc_inverse2', 'HouseAge_2', 'HouseAge_3', 'HouseAge_4', 'HouseAge_inverse', 'HouseAge_inverse2', 'AveRooms_2', 'AveRooms_3', 'AveRooms_4', 'AveRooms_inverse', 'AveRooms_inverse2', 'AveBedrms_2', 'AveBedrms_3', 'AveBedrms_4', 'AveBedrms_inverse', 'AveBedrms_inverse2', 'Population_2', 'Population_3', 'Population_4', 'Population_inverse', 'Population_inverse2', 'AveOccup_2', 'AveOccup_3', 'AveOccup_4', 'AveOccup_inverse', 'AveOccup_inverse2', 'Latitude_2', 'Latitude_3', 'Latitude_4', 'Latitude_inverse', 'Latitude_inverse2', 'Longitude_2', 'Longitude_3', 'Longitude_4', 'Longitude_inverse', 'Longitude_inverse2']


In [52]:
# Create transformer for numeric features
numeric_transformer = StandardScaler()

In [53]:
# Create transformer for categorical features
categorical_transformer = OneHotEncoder(drop='first', handle_unknown = 'ignore') # ignore if unknown categories are found in test set

In [54]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [55]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
X_train = preprocessor.fit_transform(X_train)
print('...Done.')

# Preprocessings on test set
print("Performing preprocessings on test set...")
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')

Performing preprocessings on train set...
...Done.
Performing preprocessings on test set...
...Done.


c:\Users\briic\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning:

Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros



14. Train a regression model and evaluate the performances. Are you satisfied?

In [56]:
# Train model
print("Train model...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Y_train_pred = regressor.predict(X_train)
Y_test_pred = regressor.predict(X_test)

# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))

Train model...
...Done.
R2 score on training set :  0.9534395615489626
R2 score on test set :  -7037.193761855995
